In [2]:


from __future__ import annotations

import argparse
import json
import re
import warnings
from pathlib import Path
from typing import Optional, Tuple, Dict, Any, List

import numpy as np
import pandas as pd
from scipy import signal as sp_sig
from scipy.signal import welch, iirnotch
from scipy.interpolate import interp1d

warnings.filterwarnings("ignore")

# ---------------- PROJECT PATHS ----------------
P4_ROOT_DIR = Path(r"/home/tsultan1/IROS/Data")
P4_DATASET_DIR = P4_ROOT_DIR / "_dataset_icml_v1"
P4_MANIFEST_DEFAULT = P4_DATASET_DIR / "manifest_v3.csv"

# ---------------- CONFIG ----------------
P4_VERSION = "v4.3"
P4_CACHE_SUFFIX = f".preproc.{P4_VERSION}.npz"   # -> ".preproc.v4_3.npz"


P4_TARGET_FS = 250.0
P4_FS_TOL_FRAC = 0.01  # resample if |fs/TARGET_FS - 1| > 1%

# EEG filtering
P4_EEG_BAND = (1.0, 40.0)  # Hz
P4_EEG_ORDER = 4
P4_APPLY_NOTCH_60 = False
P4_APPLY_NOTCH_50 = False
P4_NOTCH_Q = 30.0
P4_EEG_CAR_MODE = "median"  # "median" or "mean"

# Auto-notch / repair
P4_EEG_AUTO_NOTCH = True
P4_EEG_NOTCH_DB = 30.0
P4_EEG_REPAIR = "hampel"  # "none" | "hampel" | "zscore"
P4_EEG_REPAIR_WIN_MS = 120.0
P4_EEG_REPAIR_SIGMA = 8.0
P4_EEG_POLARITY_CHECK = True

# EMG envelope
P4_EMG_ENVELOPE_MODE = "hp_rect_lp"  # "hp_rect_lp" or "bp_rect_ma"
P4_EMG_HP_HZ = 20.0
P4_EMG_LP_HZ = 10.0
P4_EMG_BP = (20.0, 100.0)
P4_EMG_ORDER = 4
P4_MA_MS = 50.0

# Eye-tracking cleanup
P4_ET_INVALID_GT = 1       # validity > 1 => invalid  (0/1 valid, 2+ invalid)
P4_ET_GAP_MS = 150.0       # interpolate gaps shorter than this
P4_ET_SMOOTH_TAPS = 7      # odd integer

# Canonical channels (EEG/EMG fixed; ET discovered globally)
P4_EEG_CH = [f"EEG_Ch{i}" for i in range(1, 9)]
P4_EMG_CH = [f"EMG_Ch{i}" for i in range(1, 5)]

# ET flags (excluded from ET continuous schema)
P4_ET_FLAG_CH = [
    "ET_Fixation", "ET_Worn", "ET_Blink",
    "ET_ValidityLeftEye", "ET_ValidityRightEye",
]

# ✅ v4.2+ : always-keep ET continuous core = gaze + pupil ONLY
P4_ET_CORE_ALWAYS = [
    "ET_GazeLeftx", "ET_GazeLefty", "ET_GazeRightx", "ET_GazeRighty",
    "ET_PupilLeft", "ET_PupilRight",
]

# Caching (versioned)
P4_SAVE_CACHE = True


# Schema JSON (versioned)
P4_SCHEMA_JSON = P4_DATASET_DIR / f"et_global_cont_cols_{P4_VERSION}.json"

# Global ET presence threshold (across files)
P4_ET_PRESENCE_THR = 0.95   # override via --presence_thr

# NUMERIC-AWARE schema knobs (per-file)
P4_SCHEMA_NROWS = 4000
P4_SCHEMA_MIN_FINITE_FRAC = 0.05  # ET col must have >=5% finite to count as present

# ---------------- overlap trim + mask-aware filtering knobs ----------------
# ✅ v4.3: default OFF (Phase-3 already makes safe windows)
P4_OVERLAP_TRIM_ENABLE = False
P4_OVERLAP_GAP_FILL_S = 0.20
P4_OVERLAP_MIN_LEN_S = 3.0
P4_OVERLAP_EDGE_PAD_S = 0.00

# overlap-trim should NOT depend on ET quality by default
P4_TRIM_REQUIRE_ET_VALID = False

# Mask-aware filter: interpolate ONLY short gaps up to this duration
P4_FILTER_GAP_S = 0.15
P4_MIN_SEG_S = 0.50


# =============================================================================
#                               FILTER HELPERS
# =============================================================================
def p4x_sos_bandpass(low: float, high: float, fs: float, order: int = 4):
    nyq = fs * 0.5
    lo = max(1e-6, low / nyq)
    hi = min(0.9999, high / nyq)
    return sp_sig.butter(order, [lo, hi], btype="bandpass", output="sos")

def p4x_sos_highpass(cut: float, fs: float, order: int = 4):
    nyq = fs * 0.5
    wn = max(1e-6, cut / nyq)
    return sp_sig.butter(order, wn, btype="highpass", output="sos")

def p4x_sos_lowpass(cut: float, fs: float, order: int = 4):
    nyq = fs * 0.5
    wn = min(0.9999, cut / nyq)
    return sp_sig.butter(order, wn, btype="lowpass", output="sos")

def p4x_notch_filter_1d(y: np.ndarray, fs: float, f0: float, q: float = 30.0) -> np.ndarray:
    try:
        b, a = iirnotch(w0=f0, Q=q, fs=fs)
    except TypeError:
        b, a = sp_sig.iirnotch(w0=f0 / (fs / 2.0), Q=q)
    if y.size < 12:
        return y
    padlen = min(int(3 * max(len(a), len(b))), y.size - 1)
    return sp_sig.filtfilt(b, a, y, padlen=padlen)

def p4x_safe_sosfiltfilt_1d(y: np.ndarray, sos) -> np.ndarray:
    if y.size < 12:
        return y
    padlen = min(int(6 * sos.shape[0]), y.size - 1)
    try:
        return sp_sig.sosfiltfilt(sos, y, padlen=padlen)
    except Exception:
        try:
            return sp_sig.sosfiltfilt(sos, y, padlen=0)
        except Exception:
            return y

def p4x_moving_average(x: np.ndarray, taps: int) -> np.ndarray:
    if taps <= 1:
        return x
    k = np.ones(int(taps), dtype=float) / float(taps)
    if x.ndim == 1:
        return np.convolve(x, k, mode="same")
    return np.apply_along_axis(lambda c: np.convolve(c, k, mode="same"), 0, x)


# =============================================================================
#                               CORE HELPERS
# =============================================================================
def p4x_estimate_median_fs(t: np.ndarray) -> float:
    dt = np.diff(t)
    dt = dt[np.isfinite(dt) & (dt > 0)]
    if dt.size == 0:
        return np.nan
    return 1.0 / np.median(dt)

def p4x_resample_linear(t: np.ndarray, X: np.ndarray, target_fs: float) -> Tuple[np.ndarray, np.ndarray]:
    t0, t1 = float(t[0]), float(t[-1])
    n_out = int(round((t1 - t0) * target_fs)) + 1
    t_new = np.linspace(t0, t1, n_out)

    if X.ndim == 1:
        f = interp1d(t, X, kind="linear", bounds_error=False, fill_value=np.nan, assume_sorted=True)
        return t_new, f(t_new)

    X_new = np.zeros((n_out, X.shape[1]), dtype=float)
    for i in range(X.shape[1]):
        f = interp1d(t, X[:, i], kind="linear", bounds_error=False, fill_value=np.nan, assume_sorted=True)
        X_new[:, i] = f(t_new)
    return t_new, X_new

def p4x_pack_numeric_with_mask(df: pd.DataFrame, cols: List[str]) -> Tuple[np.ndarray, np.ndarray, List[str]]:
    out_cols, arrs, masks = [], [], []
    N = len(df)

    for c in cols:
        if c in df.columns:
            v = pd.to_numeric(df[c], errors="coerce").to_numpy()
            m = np.isfinite(v).astype(np.float32)
            v = np.nan_to_num(v, nan=0.0, posinf=0.0, neginf=0.0)
        else:
            v = np.zeros(N, dtype=float)
            m = np.zeros(N, dtype=np.float32)
        out_cols.append(c)
        arrs.append(v)
        masks.append(m)

    X = np.column_stack(arrs) if arrs else np.zeros((N, 0), dtype=float)
    M = np.column_stack(masks) if masks else np.zeros((N, 0), dtype=np.float32)
    return X, M, out_cols


# =============================================================================
#                       OVERLAP TRIM HELPERS
# =============================================================================
def p4x_fill_short_false_gaps(mask: np.ndarray, max_gap: int) -> np.ndarray:
    m = mask.astype(bool).copy()
    n = m.size
    i = 0
    while i < n:
        if m[i]:
            i += 1
            continue
        j = i
        while j < n and (not m[j]):
            j += 1
        left_true = (i - 1 >= 0) and m[i - 1]
        right_true = (j < n) and m[j]
        if left_true and right_true and (j - i) <= max_gap:
            m[i:j] = True
        i = j
    return m

def p4x_longest_true_run(mask: np.ndarray) -> Tuple[int, int]:
    m = mask.astype(bool)
    n = m.size
    best_s = best_e = 0
    i = 0
    while i < n:
        if not m[i]:
            i += 1
            continue
        j = i
        while j < n and m[j]:
            j += 1
        if (j - i) > (best_e - best_s):
            best_s, best_e = i, j
        i = j
    return best_s, best_e

def p4x_trim_to_overlap(
    t: np.ndarray,
    EEG_M: np.ndarray,
    EMG_M: np.ndarray,
    ET_any_mask: np.ndarray,
    ET_valid: np.ndarray,
    fs: float,
    require_et_valid: bool = False,
) -> Tuple[int, int, Dict[str, Any]]:
    """
    overlap = EEG(any) & EMG(any) & ET(any)
    (optionally) & (ET_valid==1) if require_et_valid=True
    """
    N = len(t)
    meta: Dict[str, Any] = {"trim_applied": False}
    if N == 0:
        return 0, 0, meta

    eeg_any = (EEG_M.sum(axis=1) > 0) if EEG_M.size else np.zeros(N, dtype=bool)
    emg_any = (EMG_M.sum(axis=1) > 0) if EMG_M.size else np.zeros(N, dtype=bool)
    et_any  = ET_any_mask.astype(bool) if ET_any_mask is not None else np.zeros(N, dtype=bool)

    overlap = eeg_any & emg_any & et_any
    if require_et_valid:
        overlap = overlap & (ET_valid.astype(int) == 1)

    gap_fill_n = max(1, int(round(P4_OVERLAP_GAP_FILL_S * fs)))
    overlap2 = p4x_fill_short_false_gaps(overlap, gap_fill_n)

    s, e = p4x_longest_true_run(overlap2)
    min_len_n = int(round(P4_OVERLAP_MIN_LEN_S * fs))
    if (e - s) < min_len_n:
        meta.update({
            "trim_applied": False,
            "trim_reason": "no_sufficient_overlap_run",
            "overlap_frac": float(np.mean(overlap)) if overlap.size else 0.0,
            "overlap2_frac": float(np.mean(overlap2)) if overlap2.size else 0.0,
            "require_et_valid": bool(require_et_valid),
        })
        return 0, N, meta

    pad_n = int(round(P4_OVERLAP_EDGE_PAD_S * fs))
    s2 = min(max(0, s + pad_n), N)
    e2 = max(min(N, e - pad_n), s2)

    meta.update({
        "trim_applied": True,
        "trim_start_idx": int(s2),
        "trim_end_idx": int(e2),
        "trim_start_time_s": float(t[s2]),
        "trim_end_time_s": float(t[e2 - 1]),
        "overlap_frac": float(np.mean(overlap)) if overlap.size else 0.0,
        "overlap2_frac": float(np.mean(overlap2)) if overlap2.size else 0.0,
        "require_et_valid": bool(require_et_valid),
    })
    return s2, e2, meta


# =============================================================================
#                    MASK-AWARE FILTERING HELPERS
# =============================================================================
def p4x_mask_aware_interp_and_segments(
    x: np.ndarray, m: np.ndarray, fs: float, gap_s: float
) -> Tuple[np.ndarray, List[Tuple[int, int]], np.ndarray]:
    N = x.size
    gap_n = max(1, int(round(gap_s * fs)))
    s = pd.Series(np.asarray(x, float).copy())
    valid = (m.astype(float) > 0.5) & np.isfinite(s.to_numpy())
    s[~valid] = np.nan

    si = s.interpolate(method="linear", limit=gap_n, limit_direction="both")
    xi = si.to_numpy()
    support = np.isfinite(xi)

    segs: List[Tuple[int, int]] = []
    i = 0
    while i < N:
        if not support[i]:
            i += 1
            continue
        j = i
        while j < N and support[j]:
            j += 1
        segs.append((i, j))
        i = j
    return xi, segs, support

def p4x_mask_aware_sosfiltfilt(
    X: np.ndarray, M: np.ndarray, fs: float, sos, gap_s: float, min_seg_s: float
) -> np.ndarray:
    if X.size == 0:
        return X.astype(np.float32)
    N, D = X.shape
    out = np.zeros((N, D), dtype=float)
    min_seg_n = max(12, int(round(min_seg_s * fs)))

    for k in range(D):
        x = X[:, k]
        m = M[:, k] if M.size else np.ones(N, dtype=float)
        xi, segs, _ = p4x_mask_aware_interp_and_segments(x, m, fs, gap_s=gap_s)

        for (s, e) in segs:
            if (e - s) < min_seg_n:
                out[s:e, k] = np.nan_to_num(xi[s:e], nan=0.0)
                continue
            y = np.nan_to_num(xi[s:e], nan=0.0)
            out[s:e, k] = p4x_safe_sosfiltfilt_1d(y, sos)

    return out.astype(np.float32)

def p4x_mask_aware_notch(
    X: np.ndarray, M: np.ndarray, fs: float, f0: float, q: float, gap_s: float, min_seg_s: float
) -> np.ndarray:
    if X.size == 0:
        return X.astype(np.float32)
    N, D = X.shape
    out = np.zeros((N, D), dtype=float)
    min_seg_n = max(12, int(round(min_seg_s * fs)))

    for k in range(D):
        x = X[:, k]
        m = M[:, k] if M.size else np.ones(N, dtype=float)
        xi, segs, _ = p4x_mask_aware_interp_and_segments(x, m, fs, gap_s=gap_s)

        for (s, e) in segs:
            if (e - s) < min_seg_n:
                out[s:e, k] = np.nan_to_num(xi[s:e], nan=0.0)
                continue
            y = np.nan_to_num(xi[s:e], nan=0.0)
            out[s:e, k] = p4x_notch_filter_1d(y, fs, f0=f0, q=q)

    return out.astype(np.float32)

def p4x_mask_aware_car(X: np.ndarray, M: np.ndarray, mode: str = "median") -> np.ndarray:
    if X.size == 0:
        return X.astype(np.float32)
    Xf = np.asarray(X, float)
    Mf = (np.asarray(M, float) > 0.5) if M.size else np.ones_like(Xf, dtype=bool)

    N, D = Xf.shape
    ref = np.zeros((N, 1), dtype=float)

    for i in range(N):
        idx = Mf[i]
        if idx.sum() < 2:
            ref[i, 0] = 0.0
            continue
        vals = Xf[i, idx]
        ref[i, 0] = np.median(vals) if mode == "median" else np.mean(vals)

    return (Xf - ref).astype(np.float32)


# =============================================================================
#                             EEG EXTRAS
# =============================================================================
def p4x_auto_notch_frequency(X: np.ndarray, fs: float, db_thresh: float = 30.0) -> Optional[float]:
    try:
        if X.size == 0:
            return None
        rms = np.sqrt(np.nanmean(X ** 2, axis=0))
        if not np.isfinite(rms).any():
            return None
        k = int(np.nanargmin(np.abs(rms - np.nanmedian(rms))))
        y = np.asarray(X[:, k], float)

        nper = min(len(y), 2048)
        if nper < 256:
            return None

        f, P = welch(y, fs=fs, nperseg=nper)
        Pdb = 10 * np.log10(P + 1e-15)

        for f0 in (60.0, 50.0):
            if f0 >= 0.45 * fs:
                continue
            band = (f >= f0 - 0.5) & (f <= f0 + 0.5)
            left = (f >= f0 - 4.0) & (f < f0 - 1.5)
            right = (f > f0 + 1.5) & (f <= f0 + 4.0)
            if not band.any() or not (left.any() and right.any()):
                continue

            peak = np.nanmax(Pdb[band])
            base = np.nanmean([np.nanmedian(Pdb[left]), np.nanmedian(Pdb[right])])
            if np.isfinite(peak) and np.isfinite(base) and (peak - base) >= float(db_thresh):
                return float(f0)
    except Exception:
        pass
    return None

def p4x_hampel_repair_1d_masked(y: np.ndarray, m: np.ndarray, fs: float, win_ms: float, n_sigma: float) -> Tuple[np.ndarray, float]:
    y = np.asarray(y, float).copy()
    m = (np.asarray(m, float) > 0.5)
    if m.sum() < 50:
        return y, 0.0

    k = max(1, int(round((win_ms / 1000.0) * fs)))
    s = pd.Series(y)
    s[~m] = np.nan

    med = s.rolling(window=2 * k + 1, center=True, min_periods=max(1, k // 2)).median()
    mad = (s - med).abs().rolling(window=2 * k + 1, center=True, min_periods=max(1, k // 2)).median()

    medv = med.to_numpy()
    sig = 1.4826 * mad.to_numpy() + 1e-12

    bad = np.zeros_like(y, dtype=bool)
    good = np.isfinite(medv) & np.isfinite(sig) & m
    bad[good] = np.abs(y[good] - medv[good]) > (n_sigma * sig[good])

    frac = float(bad[m].mean()) if m.sum() else 0.0
    if bad.any() and frac < 0.30:
        yc = y.copy()
        yc[bad] = np.nan
        idx = np.arange(y.size)
        ok = np.isfinite(yc)
        if ok.sum() >= 2:
            y[~ok] = np.interp(idx[~ok], idx[ok], yc[ok])

    return y, frac

def p4x_optional_polarity_flip_masked(y_raw: np.ndarray, y_clean: np.ndarray, m: np.ndarray) -> Tuple[np.ndarray, bool]:
    xr = np.asarray(y_raw, float)
    xc = np.asarray(y_clean, float)
    m = (np.asarray(m, float) > 0.5) & np.isfinite(xr) & np.isfinite(xc)
    if m.sum() < 200:
        return y_clean, False

    corr = np.corrcoef(xr[m], xc[m])[0, 1]
    ratio = (np.sqrt(np.mean(xc[m] ** 2)) + 1e-12) / (np.sqrt(np.mean(xr[m] ** 2)) + 1e-12)

    if corr <= -0.90 and abs(1.0 - ratio) <= 0.30:
        return (-xc), True
    return xc, False


# =============================================================================
#                         COLUMN CANONICALIZATION
# =============================================================================
def p4x_standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    mapping: Dict[Any, str] = {}

    for c in df.columns:
        s0 = str(c).strip()
        s = s0.replace(" ", "_").replace("-", "_")

        # Time
        if s.lower() in {"timestamp_seconds", "timestamps_seconds"}:
            mapping[c] = "Timestamp_seconds"
            continue
        if s.lower() in {"timestamp_ms", "timestamps_ms"}:
            mapping[c] = "Timestamp_ms"
            continue

        # EMG raw aliases like "Ch1 EMG raw"
        m = re.match(r"(?i)ch\s*([1-4])\s*emg\s*raw", s0)
        if m:
            mapping[c] = f"EMG_Ch{int(m.group(1))}"
            continue

        # EEG aliases "Ch1".."Ch8"
        m = re.match(r"(?i)^ch\s*([1-8])$", s0)
        if m:
            mapping[c] = f"EEG_Ch{int(m.group(1))}"
            continue

        # Already-canonical EMG/EEG
        m = re.match(r"(?i)^emg[_\s]*ch\s*([1-4])$", s0)
        if m:
            mapping[c] = f"EMG_Ch{int(m.group(1))}"
            continue
        m = re.match(r"(?i)^eeg[_\s]*ch\s*([1-8])$", s0)
        if m:
            mapping[c] = f"EEG_Ch{int(m.group(1))}"
            continue

        # ET: normalize a few common variants safely
        s_up = s0.strip()
        core_alias = {
            # gaze
            "ET_GazeLeftX": "ET_GazeLeftx",
            "ET_GazeLeftY": "ET_GazeLefty",
            "ET_GazeRightX": "ET_GazeRightx",
            "ET_GazeRightY": "ET_GazeRighty",
            # pupil
            "ET_PupilLeftDiameter": "ET_PupilLeft",
            "ET_PupilRightDiameter": "ET_PupilRight",
            "ET_PupilDiameterLeft": "ET_PupilLeft",
            "ET_PupilDiameterRight": "ET_PupilRight",
        }
        if s_up in core_alias:
            mapping[c] = core_alias[s_up]
            continue

        if s_up.startswith("ET_"):
            s2 = s_up
            s2 = s2.replace("LeftX", "Leftx").replace("LeftY", "Lefty")
            s2 = s2.replace("RightX", "Rightx").replace("RightY", "Righty")
            mapping[c] = s2
            continue

        mapping[c] = s0.strip()

    out = df.rename(columns=mapping)

    if "Timestamp_seconds" not in out.columns:
        if "Timestamp_ms" in out.columns:
            out["Timestamp_seconds"] = pd.to_numeric(out["Timestamp_ms"], errors="coerce") / 1000.0
        else:
            raise ValueError("Missing time: neither Timestamp_seconds nor Timestamp_ms found.")

    return out


# =============================================================================
#                    GLOBAL ET SCHEMA (NUMERIC-AWARE)
# =============================================================================
def p4x_is_et_flag_col(name: str) -> bool:
    if name in set(P4_ET_FLAG_CH):
        return True
    bad_patterns = [
        r"(?i)validity", r"(?i)worn", r"(?i)fixation", r"(?i)blink",
        r"(?i)saccade", r"(?i)event", r"(?i)flag"
    ]
    return any(re.search(p, name) for p in bad_patterns)

def p4x_extract_et_cont_from_cols(cols: List[str]) -> List[str]:
    out = []
    for c in cols:
        if not str(c).startswith("ET_"):
            continue
        if p4x_is_et_flag_col(str(c)):
            continue
        out.append(str(c))
    return out

def p4x_build_global_et_schema(
    manifest_csv: Path,
    presence_thr: float = P4_ET_PRESENCE_THR,
    nrows: int = P4_SCHEMA_NROWS,
    min_finite_frac: float = P4_SCHEMA_MIN_FINITE_FRAC,
) -> List[str]:
    man = pd.read_csv(manifest_csv)
    if "file" in man.columns:
        paths = man["file"].astype(str).tolist()
    elif "label_csv" in man.columns:
        paths = man["label_csv"].astype(str).tolist()
    else:
        raise ValueError("manifest must contain 'file' or 'label_csv' column")

    counts: Dict[str, int] = {}
    total = 0

    for p in paths:
        fp = Path(p)
        if not fp.exists():
            continue
        try:
            df = pd.read_csv(fp, low_memory=False, nrows=int(nrows))
            df = p4x_standardize_columns(df)
            cols = df.columns.tolist()

            et_cont = p4x_extract_et_cont_from_cols(cols)
            total += 1
            if not et_cont:
                continue

            present = set()
            for c in et_cont:
                v = pd.to_numeric(df[c], errors="coerce").to_numpy()
                finite_frac = float(np.isfinite(v).mean()) if v.size else 0.0
                if finite_frac >= float(min_finite_frac):
                    present.add(c)

            for c in present:
                counts[c] = counts.get(c, 0) + 1

        except Exception:
            continue

    if total == 0:
        return []

    keep = [c for c, k in counts.items() if (k / total) >= float(presence_thr)]

    # guarantee core ET continuous columns are always included
    for c in P4_ET_CORE_ALWAYS:
        if c not in keep:
            keep.append(c)

    core = [c for c in P4_ET_CORE_ALWAYS if c in keep]
    rest = sorted([c for c in keep if c not in set(P4_ET_CORE_ALWAYS)])
    return core + rest

def p4x_load_or_create_et_schema(
    manifest_csv: Path,
    schema_json: Path = P4_SCHEMA_JSON,
    presence_thr: float = P4_ET_PRESENCE_THR,
    rebuild: bool = False,
) -> List[str]:
    if schema_json.exists() and (not rebuild):
        try:
            obj = json.loads(schema_json.read_text())
            cols = obj.get("et_cont_cols", [])
            if isinstance(cols, list):
                cols = [str(x) for x in cols]
                for c in P4_ET_CORE_ALWAYS:
                    if c not in cols:
                        cols.append(c)
                core = [c for c in P4_ET_CORE_ALWAYS if c in cols]
                rest = sorted([c for c in cols if c not in set(P4_ET_CORE_ALWAYS)])
                return core + rest
        except Exception:
            pass

    cols = p4x_build_global_et_schema(
        manifest_csv,
        presence_thr=presence_thr,
        nrows=P4_SCHEMA_NROWS,
        min_finite_frac=P4_SCHEMA_MIN_FINITE_FRAC,
    )
    schema_json.write_text(json.dumps({
        "phase4_version": P4_VERSION,
        "presence_thr": float(presence_thr),
        "nrows": int(P4_SCHEMA_NROWS),
        "min_finite_frac": float(P4_SCHEMA_MIN_FINITE_FRAC),
        "et_cont_cols": cols
    }, indent=2))
    return cols


# =============================================================================
#                               ET CLEANUP
# =============================================================================
def p4x_et_interpolate_smooth(
    X: np.ndarray,
    et_valid: np.ndarray,
    fs: float,
    gap_ms: float,
    smooth_taps: int
) -> Tuple[np.ndarray, np.ndarray]:
    N, D = X.shape
    out = X.copy()
    feat_mask = np.zeros((N, D), dtype=np.float32)

    v = (et_valid.astype(int) == 1)
    lim = max(1, int(round((gap_ms / 1000.0) * fs)))
    taps = smooth_taps if smooth_taps % 2 == 1 else smooth_taps + 1

    for d in range(D):
        col = out[:, d].astype(float)
        m = np.isfinite(col) & v

        s = pd.Series(col)
        s[~m] = np.nan

        s_i = s.interpolate(method="linear", limit=lim, limit_direction="both")
        long_gap = s_i.isna().to_numpy()
        s_i = s_i.fillna(0.0)

        sm = pd.Series(s_i).rolling(window=taps, center=True, min_periods=1).mean().to_numpy()

        out[:, d] = sm
        feat_mask[:, d] = (~long_gap).astype(np.float32)

    return out, feat_mask


# =============================================================================
#                               PREPROCESSOR
# =============================================================================
def p4x_process_one_file(
    csv_path: Path,
    et_cont_cols_global: List[str],
    rebuild_cache: bool = False
) -> Dict[str, Any]:
    csv_path = Path(csv_path)
    out_npz = csv_path.with_suffix(csv_path.suffix + P4_CACHE_SUFFIX)

    if out_npz.exists() and not rebuild_cache:
        npz = np.load(out_npz, allow_pickle=True)
        return {"fs": float(npz["fs"]), "t": npz["t"], "log": json.loads(npz["log"].item())}

    df = pd.read_csv(csv_path, low_memory=False)
    df = p4x_standardize_columns(df)

    # ----- Timebase & fs (raw) -----
    t_raw = pd.to_numeric(df["Timestamp_seconds"], errors="coerce").to_numpy()
    if len(t_raw) < 4:
        raise ValueError(f"{csv_path.name}: too few samples")

    good = np.isfinite(t_raw)
    df = df.loc[good].copy()
    t_raw = t_raw[good]

    order = np.argsort(t_raw)
    df = df.iloc[order].reset_index(drop=True)
    t_raw = t_raw[order]

    _, uniq = np.unique(t_raw, return_index=True)
    df = df.iloc[np.sort(uniq)].reset_index(drop=True)
    t_raw = df["Timestamp_seconds"].to_numpy()

    fs_raw = p4x_estimate_median_fs(t_raw)
    if not np.isfinite(fs_raw) or fs_raw <= 0:
        raise ValueError(f"{csv_path.name}: invalid fs")

    # ----- Labels -----
    if "label_action" not in df.columns or "task_target" not in df.columns:
        raise ValueError(f"{csv_path.name}: missing label columns (label_action/task_target)")
    y_action = pd.to_numeric(df["label_action"], errors="coerce").fillna(0).astype(int).to_numpy()
    y_task   = pd.to_numeric(df["task_target"], errors="coerce").fillna(0).astype(int).to_numpy()

    # ----- Extract modalities -----
    EEG_X, EEG_M, EEG_cols = p4x_pack_numeric_with_mask(df, P4_EEG_CH)
    EMG_X, EMG_M, EMG_cols = p4x_pack_numeric_with_mask(df, P4_EMG_CH)

    # Ensure ET schema always includes core
    et_cols_use = list(et_cont_cols_global)
    for c in P4_ET_CORE_ALWAYS:
        if c not in et_cols_use:
            et_cols_use.insert(0, c)
    seen = set()
    et_cols_use = [c for c in et_cols_use if (c not in seen and not seen.add(c))]

    ET_X, ET_M, ET_cols = p4x_pack_numeric_with_mask(df, et_cols_use)

    # =============================================================================
    # ROBUST ET_valid (core presence = gaze+pupil only)
    # =============================================================================
    def _num(col, default=None):
        if col in df.columns:
            return pd.to_numeric(df[col], errors="coerce").to_numpy()
        return default

    vl = _num("ET_ValidityLeftEye", None)
    vr = _num("ET_ValidityRightEye", None)
    worn_raw = _num("ET_Worn", None)

    core_present = np.zeros(len(df), dtype=bool)
    for c in P4_ET_CORE_ALWAYS:
        if c in df.columns:
            v = pd.to_numeric(df[c], errors="coerce").to_numpy()
            core_present |= np.isfinite(v)

    if (vl is not None) and (vr is not None):
        validity_ok = (vl <= P4_ET_INVALID_GT) & (vr <= P4_ET_INVALID_GT)
    else:
        validity_ok = core_present.copy()

    worn_used = False
    worn_inverted = False
    if worn_raw is not None:
        worn = np.nan_to_num(worn_raw, nan=1.0)
        if np.nanstd(worn) > 1e-6:
            worn_used = True
            worn_ok = (worn >= 0.5)
        else:
            worn_ok = np.ones(len(df), dtype=bool)
    else:
        worn_ok = np.ones(len(df), dtype=bool)

    ET_valid = (validity_ok & worn_ok).astype(int)

    if (ET_valid.mean() < 0.01) and (core_present.mean() > 0.05) and (validity_ok.mean() > 0.20):
        if worn_raw is not None:
            worn = np.nan_to_num(worn_raw, nan=1.0)
            worn_inv_ok = (worn < 0.5)
            ET_valid_inv = (validity_ok & worn_inv_ok).astype(int)
            if ET_valid_inv.mean() > ET_valid.mean() + 0.10:
                ET_valid = ET_valid_inv
                worn_used = True
                worn_inverted = True
            else:
                ET_valid = validity_ok.astype(int)
                worn_used = False
                worn_inverted = False
        else:
            ET_valid = validity_ok.astype(int)

    if (ET_valid.mean() < 0.01) and (core_present.mean() > 0.05):
        ET_valid = core_present.astype(int)

    ET_valid = ET_valid.astype(int)

    ET_any_mask = (ET_M.sum(axis=1) > 0) if ET_M.size else core_present.copy()
    ET_any_mask = ET_any_mask | core_present

    # ----- Resample to TARGET_FS if needed -----
    resampled = False
    fs = float(fs_raw)
    t = t_raw.copy()

    if abs(fs / P4_TARGET_FS - 1.0) > P4_FS_TOL_FRAC:
        all_cols = EEG_cols + EMG_cols + ET_cols
        X_all = np.column_stack([EEG_X, EMG_X, ET_X]) if all_cols else np.zeros((len(t), 0))
        t_new, X_all = p4x_resample_linear(t, X_all, P4_TARGET_FS)

        t_old = t.copy()

        def rs_nearest_1d(x_old: np.ndarray) -> np.ndarray:
            f = interp1d(t_old, x_old, kind="nearest", bounds_error=False,
                        fill_value=(x_old[0], x_old[-1]), assume_sorted=True)
            return f(t_new)

        def rs_nearest_mask(m_old: np.ndarray) -> np.ndarray:
            if m_old.size == 0:
                return m_old
            outm = np.zeros((len(t_new), m_old.shape[1]), dtype=np.float32)
            for j in range(m_old.shape[1]):
                f = interp1d(t_old, m_old[:, j], kind="nearest", bounds_error=False,
                            fill_value=(m_old[0, j], m_old[-1, j]), assume_sorted=True)
                outm[:, j] = (f(t_new) > 0.5).astype(np.float32)
            return outm

        EEG_M = rs_nearest_mask(EEG_M)
        EMG_M = rs_nearest_mask(EMG_M)
        ET_M  = rs_nearest_mask(ET_M)

        y_action = rs_nearest_1d(y_action).round().astype(int)
        y_task   = rs_nearest_1d(y_task).round().astype(int)
        ET_valid = (rs_nearest_1d(ET_valid) > 0.5).astype(int)
        ET_any_mask = (rs_nearest_1d(ET_any_mask.astype(int)) > 0.5)
        core_present = (rs_nearest_1d(core_present.astype(int)) > 0.5)

        n_eeg, n_emg, n_et = len(EEG_cols), len(EMG_cols), len(ET_cols)
        EEG_X = X_all[:, :n_eeg] if n_eeg else np.zeros((len(t_new), 0))
        EMG_X = X_all[:, n_eeg:n_eeg + n_emg] if n_emg else np.zeros((len(t_new), 0))
        ET_X  = X_all[:, n_eeg + n_emg:] if n_et else np.zeros((len(t_new), 0))

        fs = P4_TARGET_FS
        t = t_new
        resampled = True

    EEG_X = np.nan_to_num(EEG_X, nan=0.0, posinf=0.0, neginf=0.0)
    EMG_X = np.nan_to_num(EMG_X, nan=0.0, posinf=0.0, neginf=0.0)
    ET_X  = np.nan_to_num(ET_X,  nan=0.0, posinf=0.0, neginf=0.0)

    # =============================================================================
    # overlap trim BEFORE filtering (default OFF in v4.3)
    # =============================================================================
    trim_meta = {"trim_applied": False}
    if P4_OVERLAP_TRIM_ENABLE:
        s0, e0, trim_meta = p4x_trim_to_overlap(
            t, EEG_M, EMG_M, ET_any_mask, ET_valid, fs,
            require_et_valid=P4_TRIM_REQUIRE_ET_VALID
        )
        if trim_meta.get("trim_applied", False):
            t = t[s0:e0]
            EEG_X = EEG_X[s0:e0]
            EMG_X = EMG_X[s0:e0]
            ET_X  = ET_X[s0:e0]
            EEG_M = EEG_M[s0:e0]
            EMG_M = EMG_M[s0:e0]
            ET_M  = ET_M[s0:e0]
            ET_valid = ET_valid[s0:e0]
            ET_any_mask = ET_any_mask[s0:e0]
            core_present = core_present[s0:e0]
            y_action = y_action[s0:e0]
            y_task   = y_task[s0:e0]

    # =============================================================================
    # EEG: MASK-AWARE bandpass/notch/auto-notch/CAR/repair/polarity
    # =============================================================================
    eeg_auto_notch_hz = None
    eeg_repair_frac = []

    if EEG_X.shape[1] > 0:
        sos_bp = p4x_sos_bandpass(P4_EEG_BAND[0], P4_EEG_BAND[1], fs, P4_EEG_ORDER)
        EEG_f = p4x_mask_aware_sosfiltfilt(EEG_X, EEG_M, fs, sos_bp, gap_s=P4_FILTER_GAP_S, min_seg_s=P4_MIN_SEG_S)

        if P4_APPLY_NOTCH_60:
            EEG_f = p4x_mask_aware_notch(EEG_f, EEG_M, fs, 60.0, q=P4_NOTCH_Q, gap_s=P4_FILTER_GAP_S, min_seg_s=P4_MIN_SEG_S)
        if P4_APPLY_NOTCH_50:
            EEG_f = p4x_mask_aware_notch(EEG_f, EEG_M, fs, 50.0, q=P4_NOTCH_Q, gap_s=P4_FILTER_GAP_S, min_seg_s=P4_MIN_SEG_S)

        if P4_EEG_AUTO_NOTCH:
            f0_auto = p4x_auto_notch_frequency(EEG_f, fs, db_thresh=P4_EEG_NOTCH_DB)
            if f0_auto is not None:
                EEG_f = p4x_mask_aware_notch(EEG_f, EEG_M, fs, float(f0_auto), q=P4_NOTCH_Q, gap_s=P4_FILTER_GAP_S, min_seg_s=P4_MIN_SEG_S)
                eeg_auto_notch_hz = float(f0_auto)

        EEG_car = p4x_mask_aware_car(EEG_f, EEG_M, mode=P4_EEG_CAR_MODE)

        if P4_EEG_REPAIR != "none":
            repaired = np.empty_like(EEG_car, dtype=np.float32)
            for k in range(EEG_car.shape[1]):
                y = EEG_car[:, k].astype(float)
                m = EEG_M[:, k].astype(float)

                if P4_EEG_REPAIR == "hampel":
                    y2, fb = p4x_hampel_repair_1d_masked(y, m, fs, win_ms=P4_EEG_REPAIR_WIN_MS, n_sigma=P4_EEG_REPAIR_SIGMA)
                else:
                    y2 = y.copy()
                    mv = (m > 0.5) & np.isfinite(y2)
                    if mv.sum() < 50:
                        fb = 0.0
                    else:
                        mu = np.nanmedian(y2[mv])
                        mad = np.nanmedian(np.abs(y2[mv] - mu)) + 1e-9
                        z = (y2 - mu) / (1.4826 * mad)
                        bad = mv & (np.abs(z) > P4_EEG_REPAIR_SIGMA)
                        fb = float(bad[mv].mean()) if mv.sum() else 0.0
                        if bad.any() and fb < 0.30:
                            yc = y2.copy()
                            yc[bad] = np.nan
                            idx = np.arange(y2.size)
                            ok = np.isfinite(yc)
                            if ok.sum() >= 2:
                                y2[~ok] = np.interp(idx[~ok], idx[ok], yc[ok])

                eeg_repair_frac.append(fb)

                if P4_EEG_POLARITY_CHECK:
                    y2, _ = p4x_optional_polarity_flip_masked(EEG_X[:, k], y2, m)

                repaired[:, k] = np.nan_to_num(y2, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

            EEG_car = repaired

        EEG_mask = EEG_M.astype(np.float32)
    else:
        EEG_car = EEG_X.astype(np.float32)
        EEG_mask = EEG_M.astype(np.float32)

    # =============================================================================
    # EMG: MASK-AWARE envelope
    # =============================================================================
    if EMG_X.shape[1] > 0:
        if P4_EMG_ENVELOPE_MODE == "hp_rect_lp":
            sos_hp = p4x_sos_highpass(P4_EMG_HP_HZ, fs, order=P4_EMG_ORDER)
            hp = p4x_mask_aware_sosfiltfilt(EMG_X, EMG_M, fs, sos_hp, gap_s=P4_FILTER_GAP_S, min_seg_s=P4_MIN_SEG_S)
            rect = np.abs(hp)

            sos_lp = p4x_sos_lowpass(P4_EMG_LP_HZ, fs, order=P4_EMG_ORDER)
            EMG_env = p4x_mask_aware_sosfiltfilt(rect, EMG_M, fs, sos_lp, gap_s=P4_FILTER_GAP_S, min_seg_s=P4_MIN_SEG_S)
        else:
            sos_bp = p4x_sos_bandpass(P4_EMG_BP[0], P4_EMG_BP[1], fs, order=P4_EMG_ORDER)
            bp = p4x_mask_aware_sosfiltfilt(EMG_X, EMG_M, fs, sos_bp, gap_s=P4_FILTER_GAP_S, min_seg_s=P4_MIN_SEG_S)
            rect = np.abs(bp)

            taps = max(1, int(round((P4_MA_MS / 1000.0) * fs)))
            if taps % 2 == 0:
                taps += 1
            EMG_env = p4x_moving_average(rect, taps)

        EMG_env = np.nan_to_num(EMG_env, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
        EMG_mask = EMG_M.astype(np.float32)
    else:
        EMG_env = EMG_X.astype(np.float32)
        EMG_mask = EMG_M.astype(np.float32)

    # =============================================================================
    # ET: validity-assisted interpolation/smoothing
    # =============================================================================
    if ET_X.shape[1] > 0:
        ET_clean, ET_feat_mask = p4x_et_interpolate_smooth(
            ET_X, ET_valid, fs, gap_ms=P4_ET_GAP_MS, smooth_taps=P4_ET_SMOOTH_TAPS
        )
        ET_clean = np.nan_to_num(ET_clean, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

        if ET_M.size:
            ET_mask_raw = (ET_feat_mask * ET_M).astype(np.float32)
        else:
            ET_mask_raw = ET_feat_mask.astype(np.float32)

        ET_mask_strict = (ET_mask_raw * ET_valid[:, None].astype(np.float32)).astype(np.float32)
        ET_mask = ET_mask_strict
    else:
        ET_clean = ET_X.astype(np.float32)
        ET_mask_raw = ET_M.astype(np.float32)
        ET_mask_strict = (ET_M * ET_valid[:, None]).astype(np.float32) if ET_M.size else ET_M.astype(np.float32)
        ET_mask = ET_mask_strict

    ET_valid = np.where(np.isfinite(ET_valid), ET_valid, 0).astype(np.int16)
    y_action = np.nan_to_num(y_action, nan=0).astype(np.int64)
    y_task   = np.nan_to_num(y_task,   nan=0).astype(np.int64)

    # =============================================================================
    # Coverage metrics (v4.3: use FINAL masks)
    # =============================================================================
    eeg_cov = float(np.mean(EEG_mask)) if EEG_mask.size else 0.0
    emg_cov = float(np.mean(EMG_mask)) if EMG_mask.size else 0.0
    et_cov_raw = float(np.mean(ET_mask_raw)) if ET_mask_raw.size else 0.0
    et_cov_strict = float(np.mean(ET_mask_strict)) if ET_mask_strict.size else 0.0
    et_cov = et_cov_strict  # legacy

    # ----- Log -----
    out_log = {
        "phase4_version": P4_VERSION,
        "file": str(csv_path.resolve()),

        "n_samples_raw": int(len(t_raw)),
        "fs_raw": float(fs_raw),

        "n_samples": int(len(t)),
        "fs": float(fs),
        "resampled_to_250": bool(resampled),

        "overlap_trim": trim_meta,
        "overlap_trim_enabled": bool(P4_OVERLAP_TRIM_ENABLE),

        "eeg_cov": float(eeg_cov),
        "emg_cov": float(emg_cov),
        "et_cov":  float(et_cov),
        "et_cov_raw": float(et_cov_raw),
        "et_cov_strict": float(et_cov_strict),

        "EEG_band": P4_EEG_BAND,
        "notch_60_fixed": bool(P4_APPLY_NOTCH_60),
        "notch_50_fixed": bool(P4_APPLY_NOTCH_50),
        "EEG_auto_notch_used_hz": eeg_auto_notch_hz,
        "EEG_CAR": P4_EEG_CAR_MODE,
        "EEG_repair": P4_EEG_REPAIR,
        "EEG_repair_frac_mean": float(np.mean(eeg_repair_frac)) if eeg_repair_frac else 0.0,

        "emg_mode": P4_EMG_ENVELOPE_MODE,

        "et_schema": f"GLOBAL_CONT_SCHEMA_{P4_VERSION} (NUMERIC-AWARE) + CORE_ALWAYS(gaze+pupil)",
        "et_presence_thr": float(P4_ET_PRESENCE_THR),
        "et_schema_nrows": int(P4_SCHEMA_NROWS),
        "et_schema_min_finite_frac": float(P4_SCHEMA_MIN_FINITE_FRAC),
        "et_cont_dim": int(len(ET_cols)),
        "et_gap_ms": float(P4_ET_GAP_MS),
        "et_smooth_taps": int(P4_ET_SMOOTH_TAPS),

        "trim_require_et_valid": bool(P4_TRIM_REQUIRE_ET_VALID),

        "et_valid_frac": float(np.mean(ET_valid)) if ET_valid.size else 0.0,
        "et_core_present_frac": float(np.mean(core_present)) if core_present.size else 0.0,
        "et_worn_used": bool(worn_used),
        "et_worn_inverted": bool(worn_inverted),

        "missing_EEG": [c for c in P4_EEG_CH if c not in df.columns],
        "missing_EMG": [c for c in P4_EMG_CH if c not in df.columns],
    }

    out = {
        "fs": float(fs),
        "t": t.astype(float),

        "EEG": EEG_car,
        "EEG_mask": EEG_mask,
        "EEG_ch": EEG_cols,

        "EMG_env": EMG_env,
        "EMG_mask": EMG_mask,
        "EMG_ch": EMG_cols,

        "ET": ET_clean,
        "ET_mask": ET_mask,                 # strict (training)
        "ET_mask_raw": ET_mask_raw,         # presence+short-gap support
        "ET_mask_strict": ET_mask_strict,   # strict quality
        "ET_ch": ET_cols,

        "ET_valid": ET_valid,
        "y_action": y_action,
        "y_task": y_task,

        "eeg_cov": float(eeg_cov),
        "emg_cov": float(emg_cov),
        "et_cov":  float(et_cov),
        "et_cov_raw": float(et_cov_raw),
        "et_cov_strict": float(et_cov_strict),

        "log": out_log,
    }

    # ----- Save cache -----
    if P4_SAVE_CACHE:
        np.savez_compressed(
            out_npz,
            fs=out["fs"], t=out["t"],

            EEG=out["EEG"], EEG_mask=out["EEG_mask"], EEG_ch=np.array(out["EEG_ch"], dtype=object),
            EMG_env=out["EMG_env"], EMG_mask=out["EMG_mask"], EMG_ch=np.array(out["EMG_ch"], dtype=object),

            ET=out["ET"],
            ET_mask=np.array(out["ET_mask"], dtype=np.float32),
            ET_mask_raw=np.array(out["ET_mask_raw"], dtype=np.float32),
            ET_mask_strict=np.array(out["ET_mask_strict"], dtype=np.float32),
            ET_ch=np.array(out["ET_ch"], dtype=object),

            ET_valid=out["ET_valid"],
            y_action=out["y_action"], y_task=out["y_task"],

            eeg_cov=np.array(out["eeg_cov"], dtype=np.float32),
            emg_cov=np.array(out["emg_cov"], dtype=np.float32),

            et_cov=np.array(out["et_cov"], dtype=np.float32),
            et_cov_raw=np.array(out["et_cov_raw"], dtype=np.float32),
            et_cov_strict=np.array(out["et_cov_strict"], dtype=np.float32),

            log=json.dumps(out["log"]),
        )
        out["log"]["cache"] = str(out_npz)

    return out


# =============================================================================
#                           BATCH VIA MANIFEST
# =============================================================================
def p4x_process_manifest(
    manifest_csv: Path,
    limit_files: Optional[int] = None,
    rebuild_cache: bool = False,
    rebuild_schema: bool = False,
    presence_thr: float = P4_ET_PRESENCE_THR,
) -> None:
    manifest_csv = Path(manifest_csv)
    if not manifest_csv.exists():
        raise FileNotFoundError(f"manifest not found: {manifest_csv}")

    et_cols = p4x_load_or_create_et_schema(
        manifest_csv=manifest_csv,
        schema_json=P4_SCHEMA_JSON,
        presence_thr=presence_thr,
        rebuild=rebuild_schema,
    )
    print(f"[Phase4 {P4_VERSION}] ET global cont dim = {len(et_cols)} (thr={presence_thr})")
    print(f"[Phase4 {P4_VERSION}] ET schema file: {P4_SCHEMA_JSON}")
    print(f"[Phase4 {P4_VERSION}] overlap_trim_enabled={P4_OVERLAP_TRIM_ENABLE} require_et_valid={P4_TRIM_REQUIRE_ET_VALID}")

    man = pd.read_csv(manifest_csv)
    if "file" in man.columns:
        paths = man["file"].astype(str).tolist()
    elif "label_csv" in man.columns:
        paths = man["label_csv"].astype(str).tolist()
    else:
        raise ValueError("manifest must contain 'file' or 'label_csv' column")

    if limit_files is not None:
        paths = paths[: int(limit_files)]

    n_ok = 0
    n_fail = 0

    for i, p in enumerate(paths, 1):
        p = Path(p)
        try:
            res = p4x_process_one_file(p, et_cont_cols_global=et_cols, rebuild_cache=rebuild_cache)
            lg = res["log"]
            print(
                f"[{i}/{len(paths)}] ok: {p.name} | fs={res['fs']:.2f} N={len(res['t'])} "
                f"| trim={lg.get('overlap_trim', {}).get('trim_applied', False)} "
                f"| cov(eeg/emg/et_raw/et_strict)={lg.get('eeg_cov',0):.3f}/{lg.get('emg_cov',0):.3f}/"
                f"{lg.get('et_cov_raw',0):.3f}/{lg.get('et_cov_strict',0):.3f} "
                f"| et_valid={lg.get('et_valid_frac',0):.3f} worn_inv={lg.get('et_worn_inverted', False)}"
            )
            n_ok += 1
        except Exception as e:
            print(f"[{i}/{len(paths)}] FAIL: {p.name} | {e}")
            n_fail += 1

    print(f"[summary] ok={n_ok} fail={n_fail}")


# =============================================================================
#                                  CLI
# =============================================================================
def p4x_build_cli() -> argparse.ArgumentParser:
    ap = argparse.ArgumentParser(
        description=f"Phase-4 {P4_VERSION} deterministic preprocessing (NUMERIC-aware ET schema + robust ET_valid + mask-aware filtering + ET raw/strict masks)."
    )
    ap.add_argument("--file", type=str, help="Path to a single label CSV to preprocess")
    ap.add_argument("--manifest", type=str, help="Path to Phase-3 manifest_v3.csv")
    ap.add_argument("--limit", type=int, default=None, help="Limit number of files when using --manifest")
    ap.add_argument("--rebuild", action="store_true", help=f"Overwrite existing {P4_CACHE_SUFFIX} caches")
    ap.add_argument("--rebuild_schema", action="store_true", help="Recompute ET global schema JSON")
    ap.add_argument("--presence_thr", type=float, default=P4_ET_PRESENCE_THR, help="ET presence threshold across files (default 0.95)")
    return ap


if __name__ == "__main__":
    args, _ = p4x_build_cli().parse_known_args()
    if args.file:
        manifest = Path(args.manifest) if args.manifest else P4_MANIFEST_DEFAULT
        et_cols = p4x_load_or_create_et_schema(
            manifest_csv=manifest,
            schema_json=P4_SCHEMA_JSON,
            presence_thr=float(args.presence_thr),
            rebuild=bool(args.rebuild_schema),
        )
        out = p4x_process_one_file(Path(args.file), et_cont_cols_global=et_cols, rebuild_cache=args.rebuild)
        print(json.dumps(out["log"], indent=2))
    else:
        manifest = Path(args.manifest) if args.manifest else P4_MANIFEST_DEFAULT
        p4x_process_manifest(
            manifest_csv=manifest,
            limit_files=args.limit,
            rebuild_cache=args.rebuild,
            rebuild_schema=args.rebuild_schema,
            presence_thr=float(args.presence_thr),
        )


[Phase4 v4.3] ET global cont dim = 15 (thr=0.95)
[Phase4 v4.3] ET schema file: /home/tsultan1/IROS/Data/_dataset_icml_v1/et_global_cont_cols_v4.3.json
[Phase4 v4.3] overlap_trim_enabled=False require_et_valid=False
[1/2486] ok: 001_T03_synchronized_corrected_icml_consensus_labels.csv | fs=250.00 N=3060 | trim=False | cov(eeg/emg/et_raw/et_strict)=0.997/0.998/0.864/0.149 | et_valid=0.149 worn_inv=False
[2/2486] ok: 002_T516_synchronized_corrected_icml_consensus_labels.csv | fs=250.00 N=2588 | trim=False | cov(eeg/emg/et_raw/et_strict)=0.997/0.995/1.000/1.000 | et_valid=1.000 worn_inv=False
[3/2486] ok: 003_T515_synchronized_corrected_icml_consensus_labels.csv | fs=250.00 N=3178 | trim=False | cov(eeg/emg/et_raw/et_strict)=0.998/0.999/1.000/1.000 | et_valid=1.000 worn_inv=False
[4/2486] ok: 004_T514_synchronized_corrected_icml_consensus_labels.csv | fs=250.00 N=2728 | trim=False | cov(eeg/emg/et_raw/et_strict)=0.998/0.997/0.845/0.125 | et_valid=0.125 worn_inv=False
[5/2486] ok: 005_T513_